# 📈 NSE Stock Performance Dashboard (2022–2024)
**Author:** Hruthvik HS | **Tools:** Python, Pandas, NumPy, Matplotlib
> 3-year analysis of 5 NSE blue-chip stocks — cumulative returns, volatility, Sharpe ratio, moving averages, and correlation matrix.

In [ ]:
!pip install pandas numpy matplotlib --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ Libraries loaded")

## 📊 Step 1 — Generate Stock Data
Realistic simulated OHLCV data for 5 NSE blue-chip stocks over 3 years (782 trading days).
> To use live data instead, run the `live_version.py` script which uses yfinance to fetch real NSE prices.

In [ ]:
STOCKS = {
    'RELIANCE': {'start':2420,'trend':0.00035,'vol':0.014},
    'TCS':      {'start':3580,'trend':0.00028,'vol':0.012},
    'INFY':     {'start':1480,'trend':0.00022,'vol':0.013},
    'HDFCBANK': {'start':1620,'trend':0.00018,'vol':0.011},
    'WIPRO':    {'start':420, 'trend':0.00015,'vol':0.015},
}
dates = pd.date_range(start='2022-01-01', end='2024-12-31', freq='B')
all_data = []
for ticker, params in STOCKS.items():
    price = params['start']; prices = []
    for i in range(len(dates)):
        shock = np.random.normal(params['trend'], params['vol'])
        if dates[i].month==3 and dates[i].year==2022: shock -= 0.003
        if dates[i].month==6 and dates[i].year==2022: shock -= 0.004
        if dates[i].month==2 and dates[i].year==2023: shock += 0.005
        price = price*(1+shock); prices.append(round(price,2))
    df = pd.DataFrame({'Date':dates,'Ticker':ticker,'Close':prices})
    all_data.append(df)

data   = pd.concat(all_data, ignore_index=True)
pivot  = data.pivot(index='Date', columns='Ticker', values='Close')
returns     = pivot.pct_change().dropna()
cum_returns = (1+returns).cumprod()-1
volatility  = returns.rolling(30).std()*np.sqrt(252)*100
total_return= ((pivot.iloc[-1]-pivot.iloc[0])/pivot.iloc[0]*100).round(2)
rf          = 0.065/252
sharpe      = ((returns.mean()-rf)/returns.std()*np.sqrt(252)).round(2)
def max_dd(s): roll_max=s.cummax(); return round(((s-roll_max)/roll_max).min()*100,2)
drawdowns   = {t: max_dd(pivot[t]) for t in STOCKS}
print(f"✅ {len(pivot)} trading days loaded")
print("\nTotal Returns:")
for t,r in total_return.items(): print(f"  {t}: {r:+.2f}%")

## 🎨 Step 2 — Dashboard

In [ ]:
COLORS = {'RELIANCE':'#E63946','TCS':'#2196F3','INFY':'#4CAF50','HDFCBANK':'#FF9800','WIPRO':'#9C27B0'}
BG='#0D1117'; PANEL='#161B22'; TEXT='#E6EDF3'; SUB='#8B949E'; GREEN='#3FB950'; RED='#F85149'

fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor=BG)
fig.suptitle('NSE STOCK PERFORMANCE DASHBOARD — 2022 to 2024',
             fontsize=16, fontweight='bold', color=TEXT, fontfamily='monospace')

# Cumulative returns
ax1 = axes[0,0]; ax1.set_facecolor(PANEL)
for t in STOCKS: ax1.plot(cum_returns.index, cum_returns[t]*100, color=COLORS[t], linewidth=1.8, label=t)
ax1.axhline(0, color=SUB, linewidth=0.8, linestyle='--', alpha=0.5)
ax1.set_title('Cumulative Returns (%)', color=TEXT, fontsize=11, fontweight='bold')
ax1.legend(facecolor=PANEL, edgecolor='#30363D', labelcolor=TEXT, fontsize=8)
ax1.tick_params(colors=SUB); ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for sp in ax1.spines.values(): sp.set_color('#30363D')
ax1.grid(axis='y', alpha=0.12, color=SUB)

# Volatility
ax2 = axes[0,1]; ax2.set_facecolor(PANEL)
for t in STOCKS: ax2.plot(volatility.index, volatility[t], color=COLORS[t], linewidth=1.2, label=t, alpha=0.85)
ax2.set_title('30-Day Rolling Volatility (%)', color=TEXT, fontsize=11, fontweight='bold')
ax2.legend(facecolor=PANEL, edgecolor='#30363D', labelcolor=TEXT, fontsize=8)
ax2.tick_params(colors=SUB); ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for sp in ax2.spines.values(): sp.set_color('#30363D')
ax2.grid(axis='y', alpha=0.12, color=SUB)

# Total return bar
ax3 = axes[0,2]; ax3.set_facecolor(PANEL)
vals = list(total_return.values); tickers = list(total_return.index)
bc = [GREEN if v>=0 else RED for v in vals]
bars3 = ax3.bar(tickers, vals, color=bc, width=0.55, edgecolor='none')
for bar,val in zip(bars3, vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(0.5 if val>=0 else -2),
             f'{val:+.1f}%', ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax3.set_title('Total Return (2022–2024)', color=TEXT, fontsize=11, fontweight='bold')
ax3.axhline(0, color=SUB, linewidth=0.8)
ax3.tick_params(colors=SUB); ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for sp in ax3.spines.values(): sp.set_color('#30363D')
ax3.grid(axis='y', alpha=0.12, color=SUB)

# Price + MAs for INFY
ax4 = axes[1,0]; ax4.set_facecolor(PANEL)
s = pivot['INFY']
ax4.plot(s.index, s, color=COLORS['INFY'], linewidth=1.2, label='INFY Price', alpha=0.9)
ax4.plot(s.index, s.rolling(20).mean(), color='#FFD700', linewidth=1, linestyle='--', label='MA20')
ax4.plot(s.index, s.rolling(50).mean(), color='#00BCD4', linewidth=1, linestyle='--', label='MA50')
ax4.plot(s.index, s.rolling(200).mean(), color='#FF5722', linewidth=1, linestyle='--', label='MA200')
ax4.set_title('INFY — Price & Moving Averages', color=TEXT, fontsize=11, fontweight='bold')
ax4.legend(facecolor=PANEL, edgecolor='#30363D', labelcolor=TEXT, fontsize=8)
ax4.tick_params(colors=SUB); ax4.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0f'))
for sp in ax4.spines.values(): sp.set_color('#30363D')
ax4.grid(axis='y', alpha=0.12, color=SUB)

# Correlation heatmap
ax5 = axes[1,1]; ax5.set_facecolor(PANEL)
corr = returns.corr(); tlist = list(STOCKS.keys())
im = ax5.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax5.set_xticks(range(len(tlist))); ax5.set_yticks(range(len(tlist)))
ax5.set_xticklabels(tlist, color=SUB, fontsize=8, rotation=30)
ax5.set_yticklabels(tlist, color=SUB, fontsize=8)
for i in range(len(tlist)):
    for j in range(len(tlist)):
        ax5.text(j,i,f'{corr.values[i,j]:.2f}',ha='center',va='center',fontsize=8,
                 color='black' if 0.3<corr.values[i,j]<0.9 else TEXT, fontweight='bold')
ax5.set_title('Return Correlation Matrix', color=TEXT, fontsize=11, fontweight='bold')
for sp in ax5.spines.values(): sp.set_color('#30363D')

# Sharpe ratio bar
ax6 = axes[1,2]; ax6.set_facecolor(PANEL)
sh_vals = [sharpe[t] for t in STOCKS]; sh_tickers = list(STOCKS.keys())
bc6 = [GREEN if v>=0 else RED for v in sh_vals]
bars6 = ax6.bar(sh_tickers, sh_vals, color=bc6, width=0.55, edgecolor='none')
for bar,val in zip(bars6, sh_vals):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+(0.02 if val>=0 else -0.05),
             f'{val:.2f}', ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax6.set_title('Sharpe Ratio (Risk-Adjusted Return)', color=TEXT, fontsize=11, fontweight='bold')
ax6.axhline(0, color=SUB, linewidth=0.8)
ax6.tick_params(colors=SUB)
for sp in ax6.spines.values(): sp.set_color('#30363D')
ax6.grid(axis='y', alpha=0.12, color=SUB)

plt.tight_layout()
plt.savefig('stock_dashboard.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()
print("✅ Dashboard generated!")

## 💡 Step 3 — Key Metrics Summary

In [ ]:
print("=" * 65)
print("          NSE STOCK PERFORMANCE — KEY METRICS (2022–2024)")
print("=" * 65)
print(f"  {'Stock':<12} {'Total Return':>14} {'Sharpe Ratio':>14} {'Max Drawdown':>14} {'Ann. Vol':>10}")
print("-" * 65)
for t in STOCKS:
    ann_vol = returns[t].std()*np.sqrt(252)*100
    print(f"  {t:<12} {total_return[t]:>+13.2f}% {sharpe[t]:>14.2f} {drawdowns[t]:>13.2f}% {ann_vol:>9.1f}%")
print("=" * 65)
print("\n  KEY INSIGHTS:")
best_t = total_return.idxmax(); worst_t = total_return.idxmin()
print(f"  Best performer  : {best_t} ({total_return[best_t]:+.2f}%)")
print(f"  Worst performer : {worst_t} ({total_return[worst_t]:+.2f}%)")
print(f"  Most stable     : {volatility.mean().idxmin()} (lowest avg volatility)")
print(f"  Best risk-adj   : {sharpe.idxmax()} (Sharpe = {sharpe.max():.2f})")